# NBA Game Prediction (Modular)

This notebook runs the modular NBA pipeline in this repo.

Rules for outputs:
- Predictions are logged to SQLite only (no JSON/CSV).
- Reports are displayed inline (no HTML files).

Prereqs:
- `pip install -r requirements.txt`
- Internet access for `nba_api` fetches

In [1]:
from __future__ import annotations

import sys
from datetime import datetime
from pathlib import Path

import pandas as pd

def find_repo_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    for p in [start] + list(start.parents):
        if (p / 'README.md').exists() and (p / 'data').exists():
            return p
    return start

ROOT = find_repo_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

DB_PATH = str(ROOT / 'sports_analytics.db')
MODEL_DIR = ROOT / 'machine_learning' / 'models'

print('ROOT:', ROOT)
print('DB_PATH:', DB_PATH)
print('MODEL_DIR:', MODEL_DIR)

ROOT: C:\Users\Windows User\My_folder\Sports_Analytics
DB_PATH: C:\Users\Windows User\My_folder\Sports_Analytics\sports_analytics.db
MODEL_DIR: C:\Users\Windows User\My_folder\Sports_Analytics\machine_learning\models


In [ ]:
from glob import glob

def _latest(glob_pattern: str) -> str | None:
    paths = sorted(glob(glob_pattern))
    return paths[-1] if paths else None

def _model_search_dirs(model_dir: Path = MODEL_DIR) -> list[Path]:
    # Support both canonical and legacy notebook-local model folders.
    candidates = [
        Path(model_dir),
        ROOT / 'basketball' / 'machine_learning' / 'models',
        Path.cwd().resolve() / 'machine_learning' / 'models',
    ]
    deduped: list[Path] = []
    seen: set[str] = set()
    for c in candidates:
        key = str(c.resolve()) if c.exists() else str(c)
        if key not in seen:
            seen.add(key)
            deduped.append(c)
    return deduped

def find_latest_model_paths(model_dir: Path = MODEL_DIR) -> dict:
    dirs = _model_search_dirs(model_dir)

    def _pick(pattern: str) -> str | None:
        for d in dirs:
            p = _latest(str(d / pattern))
            if p:
                return p
        return None

    gp = _pick('gp_*.pkl')
    lgbm_win = _pick('lgbm_win_*.pkl')
    lgbm_quantile = _pick('lgbm_quantile_*.pkl')
    elo = _pick('elo_*.pkl')
    return {
        'gp': gp,
        'lgbm_win': lgbm_win,
        'lgbm_quantile': lgbm_quantile,
        'elo': elo,
    }

print('Model search dirs:')
for d in _model_search_dirs():
    print(' -', d)

paths = find_latest_model_paths()
paths

{'gp': None, 'lgbm_win': None, 'lgbm_quantile': None, 'elo': None}

## Automatic full retrain




This notebook retrains automatically when:


- Model artifacts are missing, or


- The prediction-batch counter in SQLite (`retraining_metadata.incremental_count`) is a multiple of 7 (7, 14, 21, …).




The counter increments by 1 after each successful prediction batch (one run/day), not per game.

In [3]:
RETRAIN_EVERY_N = 7
ACCURACY_WINDOW_BATCHES = 7
POINT_TOLERANCE = 5.0

from data.database.database_handler import SportsAnalyticsDB

def _models_missing(model_paths: dict) -> list[str]:
    return [k for k, v in (model_paths or {}).items() if not v]


def _print_retrain_accuracy_snapshot(db_path: str) -> None:
    with SportsAnalyticsDB(db_path) as db:
        stats = db.get_batch_accuracy_comparison(
            window_batches=ACCURACY_WINDOW_BATCHES,
            point_tolerance=POINT_TOLERANCE,
        )

    current = stats.get('current')
    if not current:
        print('Accuracy snapshot: not enough settled games yet for comparison.')
        return

    delta = stats.get('delta') or {}

    winner_accuracy = float(current.get('winner_accuracy') or 0.0)
    point_accuracy = float(current.get('point_accuracy') or 0.0)
    spread_mae = current.get('spread_mae')

    winner_delta = delta.get('winner_accuracy')
    point_delta = delta.get('point_accuracy')
    mae_delta = delta.get('spread_mae')

    winner_msg = f'Model accuracy (winner, last {current.get("batches", 0)} batches): {winner_accuracy:.1f}%'
    if winner_delta is not None:
        winner_msg += f' ({winner_delta:+.1f} pp vs previous {ACCURACY_WINDOW_BATCHES})'
    print(winner_msg)

    point_msg = f'Point accuracy (|spread error| <= {POINT_TOLERANCE:.1f}): {point_accuracy:.1f}%'
    if point_delta is not None:
        point_msg += f' ({point_delta:+.1f} pp)'
    print(point_msg)

    if spread_mae is not None:
        mae_msg = f'Spread MAE (points): {float(spread_mae):.2f}'
        if mae_delta is not None:
            mae_msg += f' ({mae_delta:+.2f} better is positive)'
        print(mae_msg)


# Read current usage counter from DB (counts prediction *batches*)
with SportsAnalyticsDB(DB_PATH) as db:
    state = db.get_retraining_state()
    run_count = int(state.get('incremental_count') or 0)

# Check local model artifacts
paths = globals().get('paths') or find_latest_model_paths()
missing = _models_missing(paths)

# Retrain when models are missing OR when counter is a multiple of 7 (excluding 0 unless missing)
scheduled = (run_count > 0) and (run_count % RETRAIN_EVERY_N == 0)
should_retrain = bool(missing) or scheduled

print(f'Prediction-batch counter (retraining_metadata.incremental_count): {run_count}')
print('Model artifacts:', paths)

if should_retrain:
    reason = ('missing artifacts: ' + ', '.join(missing)) if missing else f'scheduled (count % {RETRAIN_EVERY_N} == 0)'
    print('Auto full retrain triggered ->', reason)
    from training.trainer import ModelTrainer

    trainer = ModelTrainer(db_path=DB_PATH)
    train_result = trainer.full_retrain(verbose=True)
    print('Trained model_version:', train_result.get('model_version'))
    paths = train_result.get('model_paths', paths)

    # Report rolling accuracy after retrain using settled games in cache.
    _print_retrain_accuracy_snapshot(DB_PATH)
else:
    print('No retrain needed.')

paths

Prediction-batch counter (retraining_metadata.incremental_count): 1
Model artifacts: {'gp': None, 'lgbm_win': None, 'lgbm_quantile': None, 'elo': None}
Auto full retrain triggered -> missing artifacts: gp, lgbm_win, lgbm_quantile, elo

FULL RETRAIN STARTED

LOADING EXTENDED TRAINING DATASET
COMPREHENSIVE DATA FETCH: 3 seasons
Seasons: 2022-23, 2023-24, 2024-25
Season type: Regular Season
Cache: enabled
  Fetching 2022-23...
    -> 2460 records
  Fetching 2023-24...
    -> 2460 records
  Fetching 2024-25...
    -> 2460 records
  Total: 7380 records, 2022-10-18 -> 2025-04-13
Cached 3690 unique games
  Building matchup features...
    3674 training rows, 2022-10-21 00:00:00 -> 2025-04-13 00:00:00
    24 feature columns
DATASET READY
Samples: 3674
Features: 24
Teams: 30

  Training GP (combined)...
    Kernel: 0.515**2 * RBF(length_scale=7.21) + 0.936**2 * Matern(length_scale=0.533, nu=1.5) + WhiteKernel(noise_level=8.62e-05)
    Log-marginal-likelihood: -4029.22
  Training LightGBM quanti

{'gp': 'machine_learning\\models\\gp_v20260314_194044.pkl',
 'lgbm_quantile': 'machine_learning\\models\\lgbm_quantile_v20260314_194044.pkl',
 'lgbm_win': 'machine_learning\\models\\lgbm_win_v20260314_194044.pkl',
 'elo': 'machine_learning\\models\\elo_v20260314_194044.pkl'}

## Generate predictions for upcoming games

This loads the latest saved models, builds leakage-safe rolling stats, predicts upcoming games, logs predictions to SQLite, and displays the HTML report inline.

In [4]:
from typing import Any

import numpy as np

from data.nba_loader import (
    fetch_nba_games,
    fetch_upcoming_games,
    get_all_nba_teams,
    get_team_latest_stats,
    prepare_prediction_features,
)
from data.feature_engineering import calculate_rolling_stats, prepare_training_data

from data.database.database_handler import SportsAnalyticsDB
from ensemble.ensemble_predictor import EnsemblePredictor
from evaluators.prediction_logger import PredictionLogger

def _to_date_str(value) -> str:
    try:
        return pd.to_datetime(value).date().isoformat()
    except Exception:
        return datetime.now().date().isoformat()

def _to_native(value: Any):
    if isinstance(value, dict):
        return {k: _to_native(v) for k, v in value.items()}
    if isinstance(value, (list, tuple)):
        return [_to_native(v) for v in value]
    if isinstance(value, np.ndarray):
        return value.tolist()
    if isinstance(value, np.generic):
        return value.item()
    return value

def _models_missing(model_paths: dict) -> list[str]:
    return [k for k, v in (model_paths or {}).items() if not v]

# 1) Ensure models exist (self-heal by retraining if needed)
paths = globals().get('paths') or find_latest_model_paths()
missing = _models_missing(paths)
if missing:
    print('Missing model artifacts:', missing)
    print('Running full retrain automatically...')
    from training.trainer import ModelTrainer

    trainer = ModelTrainer(db_path=DB_PATH)
    train_result = trainer.full_retrain(verbose=True)
    paths = train_result.get('model_paths', paths)

# 2) Load models
ep = EnsemblePredictor(model_dir=str(MODEL_DIR))
ep.load_models(
    gp_path=paths['gp'],
    lgbm_win_path=paths['lgbm_win'],
    lgbm_quantile_path=paths['lgbm_quantile'],
    elo_path=paths['elo'],
)

# 3) Historical game logs -> rolling stats -> matchup feature schema
games_df = fetch_nba_games(seasons=['2023-24', '2024-25'], season_type='Regular Season', verbose=True)
games_with_stats = calculate_rolling_stats(games_df, window=5)
matchup_df, _, feature_cols = prepare_training_data(games_with_stats, verbose=True)

# 4) Upcoming schedule
upcoming = fetch_upcoming_games(verbose=True)
team_names = get_all_nba_teams().get('names', {})

predictions = []
logged_prediction_ids = []

with SportsAnalyticsDB(DB_PATH) as db:
    model_version = (db.get_retraining_state().get('model_version') or '')
    logger = PredictionLogger(db, model_version=model_version)

    for game in upcoming:
        home_id = game.get('home_team_id')
        away_id = game.get('away_team_id')
        if not home_id or not away_id:
            continue

        home_id = int(home_id)
        away_id = int(away_id)

        home_stats = get_team_latest_stats(games_with_stats, home_id)
        away_stats = get_team_latest_stats(games_with_stats, away_id)
        if not home_stats or not away_stats:
            continue

        features_df = prepare_prediction_features(home_stats, away_stats, feature_cols)
        out = ep.predict(home_id, away_id, features_df)

        pred = {
            'game_id': game.get('game_id'),
            'game_date': _to_date_str(game.get('game_date')),
            'home_team': team_names.get(home_id, str(home_id)),
            'away_team': team_names.get(away_id, str(away_id)),
            'home_team_id': home_id,
            'away_team_id': away_id,
            'spread': float(out.get('spread', 0.0)),
            'q10': float(out.get('q10', 0.0)),
            'q90': float(out.get('q90', 0.0)),
            'uncertainty': float(out.get('uncertainty', 0.0)),
            'win_prob': float(out.get('win_prob', 0.5)),
            'confidence': str(out.get('confidence', 'LOW')),
            'model_contributions': _to_native(out.get('model_contributions', {})),
        }

        feature_snapshot = _to_native(features_df.iloc[0].to_dict())
        pred_id = logger.log_prediction(prediction=_to_native(pred), features=feature_snapshot)
        logged_prediction_ids.append(pred_id)
        predictions.append(pred)

print('Predictions generated:', len(predictions))
print('Predictions logged to SQLite:', len(logged_prediction_ids))

# 5) Increment batch counter if we actually logged predictions
if logged_prediction_ids:
    with SportsAnalyticsDB(DB_PATH) as db:
        state = db.get_retraining_state()
        prev_count = int(state.get('incremental_count') or 0)
        next_count = prev_count + 1
        db.update_retraining_state(incremental_count=next_count)
    print(f'Incremented prediction-batch counter: {prev_count} -> {next_count}')
else:
    print('No predictions logged; counter unchanged.')

pd.DataFrame([{
    'game_date': p.get('game_date'),
    'away_team': p.get('away_team'),
    'home_team': p.get('home_team'),
    'spread': p.get('spread'),
    'win_prob': p.get('win_prob'),
    'q10': p.get('q10'),
    'q90': p.get('q90'),
    'uncertainty': p.get('uncertainty'),
    'confidence': p.get('confidence'),
} for p in predictions])

  Fetching 2023-24...
    -> 2460 records
  Fetching 2024-25...
    -> 2460 records
  Total: 4920 records, 2023-10-24 -> 2025-04-13
  Building matchup features...
    2445 training rows, 2023-10-26 00:00:00 -> 2025-04-13 00:00:00
    24 feature columns
  Found 7 upcoming games
Predictions generated: 7
Predictions logged to SQLite: 7
Incremented prediction-batch counter: 0 -> 1


,game_date,away_team,home_team,spread,win_prob,q10,q90,uncertainty,confidence
0,2026-03-15,Denver Nuggets,Los Angeles Lakers,3.39,0.5697,-14.08,18.09,8.0442,MEDIUM
1,2026-03-15,Sacramento Kings,Los Angeles Clippers,3.12,0.5634,-13.32,18.70,8.0055,MEDIUM
2,2026-03-14,Brooklyn Nets,Philadelphia 76ers,3.53,0.5899,-14.25,19.32,8.3925,MEDIUM
3,2026-03-14,Milwaukee Bucks,Atlanta Hawks,-1.36,0.4662,-18.62,15.47,8.5224,MEDIUM
4,2026-03-14,Charlotte Hornets,San Antonio Spurs,3.48,0.5874,-12.93,18.87,7.9501,MEDIUM
5,2026-03-14,Washington Wizards,Boston Celtics,10.33,0.7326,-9.57,23.80,8.3431,MEDIUM
6,2026-03-15,Orlando Magic,Miami Heat,1.25,0.5271,-14.91,19.10,8.5006,MEDIUM


In [5]:
from IPython.display import display, HTML

from reports.html_report import generate as generate_html

ts = datetime.now().strftime('%Y-%m-%d %H:%M')
title = f'NBA Predictions ({ts})'

display(HTML(generate_html(predictions, title=title)))

#,Date,Matchup,Favored,Spread,Win Prob,Interval,Uncertainty,Confidence
1,2026-03-15,Denver Nuggets @ Los Angeles Lakers,Los Angeles Lakers,3.39,57.0%,"[-14.08, 18.09]",8.044,MEDIUM
2,2026-03-15,Sacramento Kings @ Los Angeles Clippers,Los Angeles Clippers,3.12,56.3%,"[-13.32, 18.70]",8.005,MEDIUM
3,2026-03-14,Brooklyn Nets @ Philadelphia 76ers,Philadelphia 76ers,3.53,59.0%,"[-14.25, 19.32]",8.393,MEDIUM
4,2026-03-14,Milwaukee Bucks @ Atlanta Hawks,Milwaukee Bucks,-1.36,46.6%,"[-18.62, 15.47]",8.522,MEDIUM
5,2026-03-14,Charlotte Hornets @ San Antonio Spurs,San Antonio Spurs,3.48,58.7%,"[-12.93, 18.87]",7.950,MEDIUM
6,2026-03-14,Washington Wizards @ Boston Celtics,Boston Celtics,10.33,73.3%,"[-9.57, 23.80]",8.343,MEDIUM
7,2026-03-15,Orlando Magic @ Miami Heat,Miami Heat,1.25,52.7%,"[-14.91, 19.10]",8.501,MEDIUM
